In [ ]:
import json, os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path(r"C:\Users\Admin\lungprints")
RES_DIR      = PROJECT_ROOT / "results"
FIG_DIR      = RES_DIR / "figures"
MODEL_DIR    = RES_DIR / "models"

SEED = 42
np.random.seed(SEED)
print("Results dir:", RES_DIR)

In [ ]:
# check which result files exist
FILES = {
    "splits":              RES_DIR / "splits.json",
    "val_comparison":      RES_DIR / "04_model_comparison.csv",
    "best_selection":      RES_DIR / "04_best_model_selection.json",
    "test_evaluation":     RES_DIR / "06_test_evaluation.json",
    "test_comparison":     RES_DIR / "06_test_comparison.csv",
    "gradcam_notes":       RES_DIR / "05_gradcam_notes.txt",
    "gradcam_meta":        RES_DIR / "05_gradcam_meta.json",
    "baseline_val":        RES_DIR / "baseline_cnn_results.json",
    "densenet_val":        RES_DIR / "densenet121_results.json",
    "efficientnet_val":    RES_DIR / "efficientnetb0_results.json",
}

print("Available result files:")
for name, path in FILES.items():
    status = "✓" if path.exists() else "✗"
    print(f"  [{status}] {name:20s} {path.name}")

In [ ]:
# load validation comparisons
if FILES["val_comparison"].exists():
    df_val = pd.read_csv(FILES["val_comparison"])
    print("Validation-set comparison:")
    print(df_val.to_string(index=False))
else:
    df_val = None
    print("No validation comparison file found.")

In [ ]:
# load test evaluations
if FILES["test_evaluation"].exists():
    with open(FILES["test_evaluation"]) as f:
        test_eval = json.load(f)

    best_model = test_eval["best_model"]
    print("Best model (from test):", best_model)

    df_test = pd.DataFrame(test_eval["test_comparison_table"])
    df_test = df_test.sort_values("f1", ascending=False).reset_index(drop=True)
    df_test_disp = df_test.copy()
    for c in ["accuracy","precision","recall","f1","roc_auc","pr_auc"]:
        df_test_disp[c] = df_test_disp[c].round(4)

    print("\nTest-set comparison:")
    print(df_test_disp.to_string(index=False))
else:
    test_eval = None
    df_test = None
    best_model = None
    print("No test evaluation file found — run Notebook 06 first.")

In [ ]:
# load grad-cam metadata
if FILES["gradcam_meta"].exists():
    with open(FILES["gradcam_meta"]) as f:
        gradcam_meta = json.load(f)
    print("Grad-CAM model:", gradcam_meta["model"])
    print("Grad-CAM samples:", {k: len(v) for k, v in gradcam_meta["samples"].items()})
else:
    gradcam_meta = None
    print("No Grad-CAM metadata found.")

In [ ]:
# final summary table 
if df_val is not None and df_test is not None:
    # Merge validation and test on model name
    merged = df_val.merge(df_test, on="model", suffixes=("_val", "_test"))
    cols = ["model", "f1_val", "f1_test", "recall_val", "recall_test",
            "roc_auc_val", "roc_auc_test", "pr_auc_val", "pr_auc_test"]
    merged = merged[cols].sort_values("f1_test", ascending=False).reset_index(drop=True)
    for c in cols[1:]:
        merged[c] = merged[c].round(4)

    print("Validation vs Test Performance")
    print("=" * 90)
    print(merged.to_string(index=False))

    merged.to_csv(RES_DIR / "07_final_summary.csv", index=False)
    print("\nSaved:", RES_DIR / "07_final_summary.csv")
else:
    merged = None
    print("Insufficient data for merged table.")

In [ ]:
# final bar chart
if df_test is not None:
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    for ax, metric in zip(axes, ["f1", "recall", "roc_auc"]):
        ax.barh(df_test["model"], df_test[metric], color="#4C72B0")
        ax.set_title(f"{metric.upper()} — Test Set")
        ax.set_xlim(0, 1.05)
        for i, v in enumerate(df_test[metric]):
            ax.text(v + 0.01, i, f"{v:.3f}", va="center", fontsize=9)
        ax.grid(alpha=0.3, axis="x")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "07_final_comparison.png", dpi=150)
    plt.show()

In [ ]:
# confusion matrix recap
if test_eval is not None:
    cm = np.array(test_eval["best_model_confusion_matrix"])
    tn, fp, fn, tp = cm.ravel()

    fig, ax = plt.subplots(figsize=(5, 4))
    ax.imshow(cm, cmap="Blues")
    ax.set_xticks([0,1]); ax.set_xticklabels(["NORMAL","PNEUMONIA"])
    ax.set_yticks([0,1]); ax.set_yticklabels(["NORMAL","PNEUMONIA"])
    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm[i,j], ha="center", va="center",
                    color="white" if cm[i,j] > cm.max()/2 else "black")
    ax.set_title(f"Final Confusion Matrix — {best_model} (Test)")
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "07_final_confusion_matrix.png", dpi=150)
    plt.show()

    print(f"True Negatives : {tn}")
    print(f"False Positives: {fp}")
    print(f"False Negatives: {fn}")
    print(f"True Positives : {tp}")

In [ ]:
# final report
if test_eval is not None:
    m = test_eval["best_model_metrics"]

    report = f"""
================================================================================
LUNGPRINTS — FINAL RESULTS SUMMARY
================================================================================

PROJECT
-------
Explainable CNN for Pediatric Pneumonia Classification from Chest Radiographs

DATASET
-------
Chest X-Ray Images (Pneumonia) — Paul Mooney (Kaggle)
Total images: 5,856 (pediatric patients, ages 1–5)
Train (post-merge + re-split): 4,185
Validation: 1,047
Test (untouched): 624

MODELING APPROACH
-----------------
- 4 architectures compared: baseline CNN, DenseNet121, ResNet50, EfficientNetB0
- Transfer learning with ImageNet weights; last block fine-tuned
- Class-weighted cross-entropy loss to counter 3:1 imbalance
- Input: 96×96 grayscale, ImageNet normalization
- Optimizer: Adam, lr=3e-4, ReduceLROnPlateau
- Early stopping on validation F1

WINNING MODEL
-------------
Model:   {best_model}
Epoch:   {test_eval['best_model_best_epoch']}

TEST SET PERFORMANCE (held out, never used during training or selection)
------------------------------------------------------------------------
Accuracy:  {m['accuracy']:.4f}
Precision: {m['precision']:.4f}
Recall:    {m['recall']:.4f}
F1-score:  {m['f1']:.4f}
ROC-AUC:   {m['roc_auc']:.4f}
PR-AUC:    {m['pr_auc']:.4f}

Confusion matrix (TN, FP, FN, TP):
  [[{tn}, {fp}],
   [{fn}, {tp}]]

KEY FINDINGS
------------
1. Transfer learning outperforms the from-scratch baseline CNN.
2. DenseNet121 achieved the best balance of F1, recall, and AUC-ROC on both
   validation and test sets.
3. Class-weighted loss allowed strong recall despite the imbalanced dataset.
4. PR-AUC confirmed the model performs well under class imbalance.

EXPLAINABILITY (GRAD-CAM)
-------------------------
- Grad-CAM implemented from scratch (no external library).
- Heatmaps on true positives concentrated on lung fields.
- False positives often highlighted ribs, borders, or medical artifacts.
- Caveat: Grad-CAM shows influential regions, not clinical evidence.

LIMITATIONS
-----------
- Single-source dataset (one hospital, one age group).
- Pediatric-only; results may not generalize to adults.
- No external validation set used.
- Not intended for clinical deployment.
- Grad-CAM is model-dependent and does not confirm lesion location.

CONCLUSION
----------
This study developed and evaluated an explainable CNN framework for pediatric
pneumonia classification. Transfer learning with DenseNet121 produced the
strongest and most balanced performance. Class imbalance handling, thorough
evaluation metrics, and Grad-CAM explainability combined to create a
methodologically sound and reproducible pipeline.

================================================================================
"""
    print(report)

    with open(RES_DIR / "07_final_report.txt", "w", encoding="utf-8") as f:
        f.write(report)
    print("Saved:", RES_DIR / "07_final_report.txt")

In [ ]:
# save final summary
if test_eval is not None:
    final = {
        "project": "Lungprints",
        "best_model": best_model,
        "best_epoch": test_eval["best_model_best_epoch"],
        "test_metrics": test_eval["best_model_metrics"],
        "test_confusion_matrix": test_eval["best_model_confusion_matrix"],
        "test_comparison": test_eval["test_comparison_table"],
        "validation_comparison": (df_val.to_dict(orient="records")
                                  if df_val is not None else None),
        "gradcam": (gradcam_meta if gradcam_meta is not None else None),
    }
    with open(RES_DIR / "07_final_summary.json", "w") as f:
        json.dump(final, f, indent=2)
    print("Saved:", RES_DIR / "07_final_summary.json")

In [ ]:
# file inventory
print("Report-ready artifacts")
print("=" * 70)
for folder, pattern in [
    (RES_DIR, "*.json"),
    (RES_DIR, "*.csv"),
    (RES_DIR, "*.txt"),
    (FIG_DIR, "*.png"),
    (FIG_DIR / "gradcam", "*.png"),
]:
    if folder.exists():
        files = sorted(folder.glob(pattern))
        for f in files:
            size_kb = f.stat().st_size / 1024
            print(f"  {f.relative_to(PROJECT_ROOT)}  ({size_kb:.0f} KB)")